In [1]:
# Land-cover Transition (km2) by Emirate + Total UAE (Excel sheets)
# - Output 2 Excel files (NOT CSV, because CSV has no sheets)
#   1) *_transition_long.xlsx    : long table per sheet
#   2) *_transition_matrix.xlsx  : 7x7 matrix per sheet
# ============================================================

import ee
import geopandas as gpd
import json
import pandas as pd
import os


In [2]:
# 0) Config
# =========================
PROJECT_ID = "ee-2015893776"
AOI_SHP = r"../../data_hmq/Map/abu_dhabi_all.shp"

OUT_DIR = r"../../Part2_SustainableDevelopmentGoal_LandDegradation/Result"
os.makedirs(OUT_DIR, exist_ok=True)


In [3]:
# 1) GEE init
# =========================
ee.Initialize(project=PROJECT_ID)


In [4]:
# 2) Reclass maps
# =========================
RECLASS_MAP_C3S = {
    # Forest -> 1
    50: 1, 60: 1, 61: 1, 62: 1, 70: 1, 71: 1, 72: 1, 80: 1, 81: 1, 82: 1, 90: 1, 100: 1,
    # Grassland -> 2
    110: 2, 120: 2, 121: 2, 122: 2, 130: 2, 140: 2, 150: 2, 151: 2, 152: 2, 153: 2,
    # Cropland -> 3
    10: 3, 11: 3, 12: 3, 20: 3, 30: 3, 40: 3,
    # Wetlands -> 4
    160: 4, 170: 4, 180: 4,
    # Artificial -> 5
    190: 5,
    # Other -> 6
    200: 6, 201: 6, 202: 6, 220: 6,
    # Water -> 7
    210: 7,
    # No data -> 0
    0: 0,
}
C3S_FROM = list(RECLASS_MAP_C3S.keys())
C3S_TO   = list(RECLASS_MAP_C3S.values())

RECLASS_MAP_ESRI = {
    2: 1,    # Trees -> Forest
    11: 2,   # Rangeland -> Grassland
    5: 3,    # Crops -> Cropland
    4: 4,    # Flooded vegetation -> Wetlands
    7: 5,    # Built area -> Artificial
    8: 6,    # Bare ground -> Other
    1: 7,    # Water -> Water
    10: 0,   # Clouds -> No data
    9: 6,    # Snow/Ice -> Other
}
ESRI_FROM = list(RECLASS_MAP_ESRI.keys())
ESRI_TO   = list(RECLASS_MAP_ESRI.values())


In [5]:
# 3) Helpers
# =========================
def clean_sheet_name(s: str) -> str:
    s = str(s).strip()
    for ch in ['\\', '/', '?', '*', '[', ']', ':']:
        s = s.replace(ch, "_")
    return s[:31] if len(s) > 31 else s

def detect_emirate_field(gdf: gpd.GeoDataFrame) -> str:
    # Try common names first
    priorities = ["emirate", "emirates", "name", "region", "admin", "state", "governorate"]
    for key in priorities:
        for c in gdf.columns:
            if key in c.lower():
                return c
    # Otherwise first object column
    obj_cols = [c for c in gdf.columns if gdf[c].dtype == "object"]
    if not obj_cols:
        raise ValueError("Cannot detect Emirate/name field: no string column found. Please add a field like 'Emirate'.")
    return obj_cols[0]

def build_geometries_from_shp(aoi_shp: str):
    gdf = gpd.read_file(aoi_shp)
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)

    emirate_col = detect_emirate_field(gdf)
    gdf["__sheet__"] = gdf[emirate_col].apply(clean_sheet_name)

    # Total geometry
    fc_all = ee.FeatureCollection(json.loads(gdf.to_json()))
    geom_total = fc_all.geometry()

    # Dissolve by emirate and build per-emirate geometries
    gdf_em = gdf.dissolve(by="__sheet__", as_index=False)
    emirate_geoms = {}
    for _, row in gdf_em.iterrows():
        sheet = row["__sheet__"]
        one = gpd.GeoDataFrame([row], geometry="geometry", crs=gdf_em.crs)
        one_fc = ee.FeatureCollection(json.loads(one.to_json()))
        emirate_geoms[sheet] = one_fc.geometry()

    return gdf, emirate_col, geom_total, emirate_geoms

def get_datasets_and_bands(geom_total: ee.Geometry):
    LC_C3S  = "projects/sat-io/open-datasets/ESA/C3S-LC-L4-LCCS"
    LC_ESRI = "projects/sat-io/open-datasets/landcover/ESRI_Global-LULC_10m_TS"

    c3s_ic_all  = ee.ImageCollection(LC_C3S).filterBounds(geom_total)
    esri_ic_all = ee.ImageCollection(LC_ESRI).filterBounds(geom_total)

    c3s_bands = c3s_ic_all.first().bandNames().getInfo()
    c3s_band = "lccs_class" if "lccs_class" in c3s_bands else c3s_bands[0]

    esri_bands = esri_ic_all.first().bandNames().getInfo()
    esri_band = esri_bands[0]

    target_proj = c3s_ic_all.first().select(c3s_band).projection()
    return c3s_ic_all, esri_ic_all, c3s_band, esri_band, target_proj

def get_cls300(source: str,
               year: int,
               geom_clip: ee.Geometry,
               c3s_ic_all: ee.ImageCollection,
               esri_ic_all: ee.ImageCollection,
               c3s_band: str,
               esri_band: str,
               target_proj: ee.Projection,
               scale: int = 300) -> ee.Image:
    source = source.upper()
    if source == "C3S":
        img = (c3s_ic_all
               .filterDate(f"{year}-01-01", f"{year}-12-31")
               .first())
        return (ee.Image(img)
                .select(c3s_band)
                .clip(geom_clip)
                .remap(C3S_FROM, C3S_TO)
                .rename("cls")
                .toInt16()
                .reproject(crs=target_proj.crs(), scale=scale))

    if source == "ESRI":
        ic_y = esri_ic_all.filterDate(f"{year}-01-01", f"{year}-12-31")
        # handle missing year
        if ic_y.size().getInfo() == 0:
            raise ValueError(f"ESRI year {year} has no data in this AOI.")
        img_y = ic_y.mosaic()
        proj10 = ee.Image(ic_y.first()).select(esri_band).projection()
        img_y = img_y.setDefaultProjection(proj10)

        cls10 = img_y.select(esri_band).clip(geom_clip).toInt16()
        cls10 = cls10.setDefaultProjection(proj10)

        cls300 = (cls10
                  .reduceResolution(reducer=ee.Reducer.mode(), maxPixels=4096)
                  .reproject(crs=target_proj.crs(), scale=scale))

        return (cls300
                .remap(ESRI_FROM, ESRI_TO)
                .rename("cls")
                .toInt16()
                .reproject(crs=target_proj.crs(), scale=scale))

    raise ValueError("source must be 'C3S' or 'ESRI'")

def build_transition_code(cls_from: ee.Image, cls_to: ee.Image, drop_nochange: bool) -> ee.Image:
    f = cls_from.select("cls").toInt16()
    t = cls_to.select("cls").toInt16()

    mask = (f.gte(1).And(f.lte(7))
            .And(t.gte(1)).And(t.lte(7)))
    if drop_nochange:
        mask = mask.And(f.neq(t))

    return f.multiply(10).add(t).rename("code").updateMask(mask).toInt16()

def transition_area_km2_dict(code_img: ee.Image,
                             geom: ee.Geometry,
                             scale: int = 300,
                             tileScale: int = 8) -> ee.Dictionary:
    area_img = ee.Image.pixelArea().addBands(code_img)
    stats = area_img.reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName="code"),
        geometry=geom,
        scale=scale,
        maxPixels=1e13,
        tileScale=tileScale
    )
    groups = ee.List(stats.get("groups"))
    # {"11": km2, ...}
    return ee.Dictionary(
        groups.map(lambda d: ee.List([
            ee.Number(ee.Dictionary(d).get("code")).format(),
            ee.Number(ee.Dictionary(d).get("sum")).divide(1e6)
        ])).flatten()
    )

def area_py_to_long(area_py: dict,
                    region: str,
                    year_from: int,
                    year_to: int,
                    source_from: str,
                    source_to: str) -> pd.DataFrame:
    rows = []
    for k, v in area_py.items():
        code = int(k)
        rows.append({
            "region": region,
            "year_from": year_from,
            "year_to": year_to,
            "source_from": source_from.upper(),
            "source_to": source_to.upper(),
            "from_class": code // 10,
            "to_class": code % 10,
            "area_km2": float(v)
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["from_class", "to_class"]).reset_index(drop=True)
    return df

def long_to_matrix(df_long: pd.DataFrame) -> pd.DataFrame:
    mat = (df_long
           .pivot(index="from_class", columns="to_class", values="area_km2")
           .reindex(index=range(1, 8), columns=range(1, 8), fill_value=0.0))
    mat.index.name = "from_class"
    mat.columns.name = "to_class"
    return mat

def write_excels(long_tables: dict, matrix_tables: dict, out_dir: str, prefix: str):
    long_xlsx = os.path.join(out_dir, f"{prefix}_transition_long.xlsx")
    mat_xlsx  = os.path.join(out_dir, f"{prefix}_transition_matrix.xlsx")

    with pd.ExcelWriter(long_xlsx, engine="openpyxl") as writer:
        # Ensure Total first
        if "Total_UAE" in long_tables:
            long_tables["Total_UAE"].to_excel(writer, sheet_name="Total_UAE", index=False)
        for sheet, df_ in long_tables.items():
            if sheet == "Total_UAE":
                continue
            df_.to_excel(writer, sheet_name=sheet, index=False)

    with pd.ExcelWriter(mat_xlsx, engine="openpyxl") as writer:
        if "Total_UAE" in matrix_tables:
            matrix_tables["Total_UAE"].to_excel(writer, sheet_name="Total_UAE", index=True)
        for sheet, df_ in matrix_tables.items():
            if sheet == "Total_UAE":
                continue
            df_.to_excel(writer, sheet_name=sheet, index=True)

    return long_xlsx, mat_xlsx

def run_transition_by_emirate(aoi_shp: str,
                             year_from: int,
                             year_to: int,
                             source_from: str,
                             source_to: str,
                             drop_nochange: bool,
                             out_dir: str,
                             scale: int = 300,
                             tileScale: int = 8):
    # Geometries
    gdf, emirate_col, geom_total, emirate_geoms = build_geometries_from_shp(aoi_shp)
    print("Emirate field:", emirate_col)
    print("Emirates:", list(emirate_geoms.keys()))

    # Datasets/bands/proj
    c3s_ic_all, esri_ic_all, c3s_band, esri_band, target_proj = get_datasets_and_bands(geom_total)

    # Build classified images (clip to total is fine; code image is global on target grid)
    cls_from = get_cls300(source_from, year_from, geom_total, c3s_ic_all, esri_ic_all, c3s_band, esri_band, target_proj, scale=scale)
    cls_to   = get_cls300(source_to,   year_to,   geom_total, c3s_ic_all, esri_ic_all, c3s_band, esri_band, target_proj, scale=scale)

    # Transition code image
    code_img = build_transition_code(cls_from, cls_to, drop_nochange=drop_nochange)

    # Compute Total
    area_total = transition_area_km2_dict(code_img, geom_total, scale=scale, tileScale=tileScale).getInfo()
    df_long_total = area_py_to_long(area_total, "Total_UAE", year_from, year_to, source_from, source_to)
    df_mat_total  = long_to_matrix(df_long_total)

    long_tables = {"Total_UAE": df_long_total}
    mat_tables  = {"Total_UAE": df_mat_total}

    # Compute each emirate
    for sheet, g in emirate_geoms.items():
        area_em = transition_area_km2_dict(code_img, g, scale=scale, tileScale=tileScale).getInfo()
        df_long_em = area_py_to_long(area_em, sheet, year_from, year_to, source_from, source_to)
        df_mat_em  = long_to_matrix(df_long_em)
        long_tables[sheet] = df_long_em
        mat_tables[sheet]  = df_mat_em
        print("Done:", sheet)

    prefix = f"{source_from.upper()}{year_from}_to_{source_to.upper()}{year_to}"
    long_xlsx, mat_xlsx = write_excels(long_tables, mat_tables, out_dir, prefix)
    print("Saved:", long_xlsx)
    print("Saved:", mat_xlsx)

    return long_tables, mat_tables, long_xlsx, mat_xlsx


In [6]:
DROP_NOCHANGE = False   # True -> keep only changes (49 -> 42)
SCALE = 300
TILESCALE = 8

In [19]:
# 4) Run
# =========================


for year in range(2000, 2015):
    YEAR_FROM = year
    YEAR_TO = year + 1
    print(f"Processing {YEAR_FROM} to {YEAR_TO}...")

    long_tables, mat_tables, long_xlsx, mat_xlsx = run_transition_by_emirate(
        aoi_shp=AOI_SHP,
        year_from=YEAR_FROM,
        year_to=YEAR_TO,
        source_from="C3S",
        source_to="C3S",
        drop_nochange=DROP_NOCHANGE,
        out_dir=OUT_DIR,
        scale=SCALE,
        tileScale=TILESCALE
    )


Processing C3S2000 to C3S2001...
Emirate field: NAME
Emirates: ['Abu Dhabi Region', 'Ajman', 'Al - Ain Region', 'Al - Dhafra Region', 'Al - Fujairah', 'Dubai - Sector I', 'Dubai - Sector II', 'Dubai - Sector III', 'Dubai - Sector IV', 'Dubai - Sector IX', 'Dubai - Sector V', 'Dubai - Sector VI', 'Dubai - Sector VII', 'Dubai - Sector VIII', 'Ras al - Khaimah', 'Sharjah', 'Umm al - Qiwain']
Done: Abu Dhabi Region
Done: Ajman
Done: Al - Ain Region
Done: Al - Dhafra Region
Done: Al - Fujairah
Done: Dubai - Sector I
Done: Dubai - Sector II
Done: Dubai - Sector III
Done: Dubai - Sector IV
Done: Dubai - Sector IX
Done: Dubai - Sector V
Done: Dubai - Sector VI
Done: Dubai - Sector VII
Done: Dubai - Sector VIII
Done: Ras al - Khaimah
Done: Sharjah
Done: Umm al - Qiwain
Saved: ../../Part2_SustainableDevelopmentGoal_LandDegradation/Result\C3S2000_to_C3S2001_transition_long.xlsx
Saved: ../../Part2_SustainableDevelopmentGoal_LandDegradation/Result\C3S2000_to_C3S2001_transition_matrix.xlsx
Processin

In [20]:
long_tables, mat_tables, long_xlsx, mat_xlsx = run_transition_by_emirate(
        aoi_shp=AOI_SHP,
        year_from=2000,
        year_to=2015,
        source_from="C3S",
        source_to="C3S",
        drop_nochange=DROP_NOCHANGE,
        out_dir=OUT_DIR,
        scale=SCALE,
        tileScale=TILESCALE
    )

Emirate field: NAME
Emirates: ['Abu Dhabi Region', 'Ajman', 'Al - Ain Region', 'Al - Dhafra Region', 'Al - Fujairah', 'Dubai - Sector I', 'Dubai - Sector II', 'Dubai - Sector III', 'Dubai - Sector IV', 'Dubai - Sector IX', 'Dubai - Sector V', 'Dubai - Sector VI', 'Dubai - Sector VII', 'Dubai - Sector VIII', 'Ras al - Khaimah', 'Sharjah', 'Umm al - Qiwain']
Done: Abu Dhabi Region
Done: Ajman
Done: Al - Ain Region
Done: Al - Dhafra Region
Done: Al - Fujairah
Done: Dubai - Sector I
Done: Dubai - Sector II
Done: Dubai - Sector III
Done: Dubai - Sector IV
Done: Dubai - Sector IX
Done: Dubai - Sector V
Done: Dubai - Sector VI
Done: Dubai - Sector VII
Done: Dubai - Sector VIII
Done: Ras al - Khaimah
Done: Sharjah
Done: Umm al - Qiwain
Saved: ../../Part2_SustainableDevelopmentGoal_LandDegradation/Result\C3S2000_to_C3S2015_transition_long.xlsx
Saved: ../../Part2_SustainableDevelopmentGoal_LandDegradation/Result\C3S2000_to_C3S2015_transition_matrix.xlsx


In [7]:
for year in range(2017, 2024):
    YEAR_FROM = year
    YEAR_TO = year + 1
    print(f"Processing {YEAR_FROM} to {YEAR_TO}...")

    long_tables, mat_tables, long_xlsx, mat_xlsx = run_transition_by_emirate(
        aoi_shp=AOI_SHP,
        year_from=YEAR_FROM,
        year_to=YEAR_TO,
        source_from="ESRI",
        source_to="ESRI",
        drop_nochange=DROP_NOCHANGE,
        out_dir=OUT_DIR,
        scale=SCALE,
        tileScale=TILESCALE
    )

Processing 2017 to 2018...
Emirate field: NAME
Emirates: ['Abu Dhabi Region', 'Ajman', 'Al - Ain Region', 'Al - Dhafra Region', 'Al - Fujairah', 'Dubai - Sector I', 'Dubai - Sector II', 'Dubai - Sector III', 'Dubai - Sector IV', 'Dubai - Sector IX', 'Dubai - Sector V', 'Dubai - Sector VI', 'Dubai - Sector VII', 'Dubai - Sector VIII', 'Ras al - Khaimah', 'Sharjah', 'Umm al - Qiwain']


KeyboardInterrupt: 

In [8]:
for year in range(2019, 2025):
    long_tables, mat_tables, long_xlsx, mat_xlsx = run_transition_by_emirate(
            aoi_shp=AOI_SHP,
            year_from=2018,
            year_to=year,
            source_from="ESRI",
            source_to="ESRI",
            drop_nochange=DROP_NOCHANGE,
            out_dir=OUT_DIR,
            scale=SCALE,
            tileScale=TILESCALE
        )

Emirate field: NAME
Emirates: ['Abu Dhabi Region', 'Ajman', 'Al - Ain Region', 'Al - Dhafra Region', 'Al - Fujairah', 'Dubai - Sector I', 'Dubai - Sector II', 'Dubai - Sector III', 'Dubai - Sector IV', 'Dubai - Sector IX', 'Dubai - Sector V', 'Dubai - Sector VI', 'Dubai - Sector VII', 'Dubai - Sector VIII', 'Ras al - Khaimah', 'Sharjah', 'Umm al - Qiwain']
Done: Abu Dhabi Region
Done: Ajman
Done: Al - Ain Region
Done: Al - Dhafra Region
Done: Al - Fujairah
Done: Dubai - Sector I
Done: Dubai - Sector II
Done: Dubai - Sector III
Done: Dubai - Sector IV
Done: Dubai - Sector IX
Done: Dubai - Sector V
Done: Dubai - Sector VI
Done: Dubai - Sector VII
Done: Dubai - Sector VIII
Done: Ras al - Khaimah
Done: Sharjah
Done: Umm al - Qiwain
Saved: ../../Part2_SustainableDevelopmentGoal_LandDegradation/Result\ESRI2018_to_ESRI2019_transition_long.xlsx
Saved: ../../Part2_SustainableDevelopmentGoal_LandDegradation/Result\ESRI2018_to_ESRI2019_transition_matrix.xlsx
Emirate field: NAME
Emirates: ['Abu Dh